# 2. Preprocessing — Run Once, Use Everywhere

**Objective:** Clean, normalize, encode, and scale ALL datasets from `/input`, then save the
processed artifacts to `/processed`. Downstream notebooks (`3_modeling`, `4_analysis`) load
these artifacts directly — this notebook should never need to be re-run unless the raw data changes.

### Pipeline
1. Scan all `input/{instrument}/{feature_type}/` directories
2. Drop rows with NaN targets, normalize `part_song` casing
3. Fit a **global** `LabelEncoder` across all datasets (consistent class indices)
4. Apply `StandardScaler` per instrument×feature combination
5. Group rows into per-track sequences (variable length)
6. Save everything into a single `processed/master_preprocessed.pkl`

> **Critical:** Run this notebook **once**. All subsequent experiments read from the saved file.

In [1]:
# ============================================================
# Cell 1 — Imports & Paths
# ============================================================
import os
import re
import pickle
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Paths ---
BASE_INPUT = Path('./input')
PROCESSED_DIR = Path('./processed')
PROCESSED_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / 'master_preprocessed.pkl'

# --- Schema constants ---
META_COLS = [
    'track_id', 'part_song', 'chord_absolute',
    'roman_numeral', 'start_time', 'end_time',
]
TARGET_COL = 'roman_numeral'
TIME_COL = 'start_time'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Input  : {BASE_INPUT.resolve()}')
print(f'Output : {OUTPUT_FILE.resolve()}')

Input  : /Users/brianashari18/Tugas Akhir/ta-model/input
Output : /Users/brianashari18/Tugas Akhir/ta-model/processed/master_preprocessed.pkl


In [2]:
# ============================================================
# Cell 2 — Discover All Datasets
# ============================================================
def discover_datasets(base: Path) -> List[Tuple[str, str, Path]]:
    """Walk input directory and return list of (instrument, feature_type, file_path)."""
    datasets = []
    for inst_dir in sorted(base.iterdir()):
        if not inst_dir.is_dir() or inst_dir.name.startswith('.'):
            continue
        for feat_dir in sorted(inst_dir.iterdir()):
            if not feat_dir.is_dir():
                continue
            feat_name = feat_dir.name
            pkl_path = feat_dir / f'extracted_features_{feat_name}.pkl'
            csv_path = feat_dir / f'extracted_features_{feat_name}.csv'
            if pkl_path.exists():
                datasets.append((inst_dir.name, feat_name, pkl_path))
            elif csv_path.exists():
                datasets.append((inst_dir.name, feat_name, csv_path))
    return datasets


all_datasets = discover_datasets(BASE_INPUT)
print(f'Discovered {len(all_datasets)} datasets:')
for inst, feat, path in all_datasets:
    print(f'  {inst:20s} / {feat:15s} → {path.name}')

Discovered 24 datasets:
  bass                 / chromagram      → extracted_features_chromagram.pkl
  bass                 / mfcc            → extracted_features_mfcc.pkl
  bass                 / mfcc_chroma     → extracted_features_mfcc_chroma.pkl
  guitar               / chromagram      → extracted_features_chromagram.pkl
  guitar               / mfcc            → extracted_features_mfcc.pkl
  guitar               / mfcc_chroma     → extracted_features_mfcc_chroma.pkl
  guitar_piano         / chromagram      → extracted_features_chromagram.pkl
  guitar_piano         / mfcc            → extracted_features_mfcc.pkl
  guitar_piano         / mfcc_chroma     → extracted_features_mfcc_chroma.pkl
  guitar_piano_bass    / chromagram      → extracted_features_chromagram.pkl
  guitar_piano_bass    / mfcc            → extracted_features_mfcc.pkl
  guitar_piano_bass    / mfcc_chroma     → extracted_features_mfcc_chroma.pkl
  no_vocals            / chromagram      → extracted_features_chromagram

In [3]:
# ============================================================
# Cell 3 — Load Utilities
# ============================================================
def load_raw(path: Path) -> pd.DataFrame:
    """Load a .pkl or .csv file into a DataFrame."""
    if path.suffix == '.pkl':
        with open(path, 'rb') as f:
            obj = pickle.load(f)
        if not isinstance(obj, pd.DataFrame):
            raise TypeError(f'Expected DataFrame, got {type(obj)}')
        return obj.copy()
    return pd.read_csv(path)


def get_feature_columns(df: pd.DataFrame) -> List[str]:
    """Return list of numeric feature column names (excluding metadata)."""
    return [c for c in df.columns if c not in META_COLS and c != 'label_idx']

In [4]:
# ============================================================
# Cell 4 — Pass 1: Collect All Unique Labels for Global Encoding
# ============================================================
all_labels = set()

for inst, feat, path in all_datasets:
    df = load_raw(path)
    valid_labels = df[TARGET_COL].dropna().astype(str).unique()
    all_labels.update(valid_labels)

# Fit global LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(sorted(all_labels))

print(f'Total unique classes: {len(label_encoder.classes_)}')
print(f'Classes: {label_encoder.classes_.tolist()}')

Total unique classes: 17
Classes: ['#v', 'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'bII', 'bVI', 'bVII', 'ii', 'iii', 'iv', 'v', 'vi', 'vii']


In [5]:
# ============================================================
# Cell 5 — Pass 2: Clean, Scale, Encode, Sequentialize
# ============================================================
def preprocess_dataset(
    df: pd.DataFrame,
    le: LabelEncoder,
) -> Dict[str, Any]:
    """
    Full preprocessing pipeline for a single instrument×feature dataset.

    Returns
    -------
    dict with keys:
        'sequences': list of {'track_id': str, 'X': ndarray(T,F), 'y': ndarray(T,)}
        'feature_columns': list of feature column names
        'n_classes': int
        'class_names': list of class name strings
        'scaler_mean': ndarray
        'scaler_scale': ndarray
    """
    df = df.copy()

    # --- 1. Drop rows with NaN target ---
    n_before = len(df)
    df = df.dropna(subset=[TARGET_COL])
    df[TARGET_COL] = df[TARGET_COL].astype(str)
    n_dropped = n_before - len(df)

    # --- 2. Normalize part_song casing ---
    if 'part_song' in df.columns:
        df['part_song'] = df['part_song'].str.lower().str.strip()

    # --- 3. Identify feature columns ---
    feat_cols = get_feature_columns(df)

    # --- 4. Standard scaling ---
    scaler = StandardScaler()
    df[feat_cols] = scaler.fit_transform(df[feat_cols].values)

    # --- 5. Encode labels ---
    df['label_idx'] = le.transform(df[TARGET_COL])

    # --- 6. Group into per-track sequences ---
    sequences = []
    for track_id, group in df.groupby('track_id', sort=False):
        group = group.sort_values(TIME_COL)
        X = group[feat_cols].values.astype(np.float32)
        y = group['label_idx'].values.astype(np.int64)
        sequences.append({'track_id': track_id, 'X': X, 'y': y})

    return {
        'sequences': sequences,
        'feature_columns': feat_cols,
        'n_classes': len(le.classes_),
        'class_names': le.classes_.tolist(),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_,
        'n_dropped_nan': n_dropped,
    }

In [6]:
# ============================================================
# Cell 6 — Run Preprocessing on All Datasets
# ============================================================
master_data: Dict[str, Dict[str, Any]] = {}
t0 = time.time()

for inst, feat, path in all_datasets:
    key = f'{inst}__{feat}'  # e.g. 'raw_audio__mfcc'
    df = load_raw(path)
    result = preprocess_dataset(df, label_encoder)

    master_data[key] = result
    n_seq = len(result['sequences'])
    n_feat = len(result['feature_columns'])
    dropped = result['n_dropped_nan']
    print(
        f'  {key:35s} │ {n_seq:3d} tracks │ '
        f'{n_feat:2d} features │ dropped NaN: {dropped}'
    )

elapsed = time.time() - t0
print(f'\nTotal datasets processed: {len(master_data)}')
print(f'Time elapsed: {elapsed:.1f}s')

  bass__chromagram                    │  50 tracks │ 24 features │ dropped NaN: 13
  bass__mfcc                          │  50 tracks │ 26 features │ dropped NaN: 13
  bass__mfcc_chroma                   │  50 tracks │ 50 features │ dropped NaN: 13
  guitar__chromagram                  │  50 tracks │ 24 features │ dropped NaN: 13
  guitar__mfcc                        │  50 tracks │ 26 features │ dropped NaN: 13
  guitar__mfcc_chroma                 │  50 tracks │ 50 features │ dropped NaN: 13
  guitar_piano__chromagram            │  50 tracks │ 24 features │ dropped NaN: 13
  guitar_piano__mfcc                  │  50 tracks │ 26 features │ dropped NaN: 13
  guitar_piano__mfcc_chroma           │  50 tracks │ 50 features │ dropped NaN: 13
  guitar_piano_bass__chromagram       │  50 tracks │ 24 features │ dropped NaN: 13
  guitar_piano_bass__mfcc             │  50 tracks │ 26 features │ dropped NaN: 13
  guitar_piano_bass__mfcc_chroma      │  50 tracks │ 50 features │ dropped NaN: 13
  no

In [7]:
# ============================================================
# Cell 7 — Save Master Artifact
# ============================================================
artifact = {
    'data': master_data,
    'label_encoder_classes': label_encoder.classes_.tolist(),
    'meta': {
        'random_seed': RANDOM_SEED,
        'meta_columns': META_COLS,
        'target_column': TARGET_COL,
    },
}

with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

file_size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f'Saved: {OUTPUT_FILE}')
print(f'Size : {file_size_mb:.2f} MB')
print(f'Keys : {list(master_data.keys())}')

Saved: processed/master_preprocessed.pkl
Size : 14.80 MB
Keys : ['bass__chromagram', 'bass__mfcc', 'bass__mfcc_chroma', 'guitar__chromagram', 'guitar__mfcc', 'guitar__mfcc_chroma', 'guitar_piano__chromagram', 'guitar_piano__mfcc', 'guitar_piano__mfcc_chroma', 'guitar_piano_bass__chromagram', 'guitar_piano_bass__mfcc', 'guitar_piano_bass__mfcc_chroma', 'no_vocals__chromagram', 'no_vocals__mfcc', 'no_vocals__mfcc_chroma', 'piano__chromagram', 'piano__mfcc', 'piano__mfcc_chroma', 'raw_audio__chromagram', 'raw_audio__mfcc', 'raw_audio__mfcc_chroma', 'vocals__chromagram', 'vocals__mfcc', 'vocals__mfcc_chroma']


In [8]:
# ============================================================
# Cell 8 — Verification: Quick Load & Sanity Check
# ============================================================
with open(OUTPUT_FILE, 'rb') as f:
    check = pickle.load(f)

print('Top-level keys:', list(check.keys()))
print('Dataset keys  :', list(check['data'].keys()))
print('Classes       :', check['label_encoder_classes'])

# Spot-check one entry
sample_key = 'raw_audio__mfcc'
sample = check['data'][sample_key]
print(f'\n--- {sample_key} ---')
print(f'  Sequences : {len(sample["sequences"])}')
print(f'  Features  : {len(sample["feature_columns"])}')
print(f'  Example X : shape={sample["sequences"][0]["X"].shape}')
print(f'  Example y : shape={sample["sequences"][0]["y"].shape}')
print(f'  X dtype   : {sample["sequences"][0]["X"].dtype}')
print(f'  y dtype   : {sample["sequences"][0]["y"].dtype}')

print('\n✅ Preprocessing complete. This notebook does not need to be run again.')

Top-level keys: ['data', 'label_encoder_classes', 'meta']
Dataset keys  : ['bass__chromagram', 'bass__mfcc', 'bass__mfcc_chroma', 'guitar__chromagram', 'guitar__mfcc', 'guitar__mfcc_chroma', 'guitar_piano__chromagram', 'guitar_piano__mfcc', 'guitar_piano__mfcc_chroma', 'guitar_piano_bass__chromagram', 'guitar_piano_bass__mfcc', 'guitar_piano_bass__mfcc_chroma', 'no_vocals__chromagram', 'no_vocals__mfcc', 'no_vocals__mfcc_chroma', 'piano__chromagram', 'piano__mfcc', 'piano__mfcc_chroma', 'raw_audio__chromagram', 'raw_audio__mfcc', 'raw_audio__mfcc_chroma', 'vocals__chromagram', 'vocals__mfcc', 'vocals__mfcc_chroma']
Classes       : ['#v', 'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'bII', 'bVI', 'bVII', 'ii', 'iii', 'iv', 'v', 'vi', 'vii']

--- raw_audio__mfcc ---
  Sequences : 50
  Features  : 26
  Example X : shape=(97, 26)
  Example y : shape=(97,)
  X dtype   : float32
  y dtype   : int64

✅ Preprocessing complete. This notebook does not need to be run again.
